In [3]:
# ============================================================
# BLOCK 1 — connect to DuckDB
# ============================================================
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))            # notebooks/ -> repo root, so `config` imports
from config import directories, logger              # project paths + loguru logger

from sqlalchemy import create_engine

DB_PATH = directories.INTERIM_DATA / "panchayat_1.duckdb"   # fresh db file
DB_URL = f"duckdb:///{DB_PATH}"                             # DuckDB dialect on this file

engine = create_engine(DB_URL)                             # everything downstream uses `engine`

with engine.connect() as conn:                             # test the connection
    logger.info(f"Connected: {conn.dialect.name} -> {DB_PATH}")

2026-08-09 07:05:09.466 | INFO     | __main__:<module>:18 - Connected: duckdb -> /home/admin_abhi/GitHub/ai-for-panchayats/data/interim/panchayat_1.duckdb


In [4]:
# ============================================================
# BLOCK 0 — load raw CSVs
# ============================================================
import pandas as pd

# planning (87 cols) — one row per planned activity
df_pl = pd.read_csv(
    "/mnt/c/Users/Admin/Desktop/Questions_Analysis/gram_panchayat_filtered (1).csv",
    low_memory=False,      # read whole columns at once -> avoids the mixed-type DtypeWarning
)

# expenditure (25 cols) — activity-wise approved cost + spend
df_ax = pd.read_csv(
    "/home/admin_abhi/GitHub/ai-for-panchayats/data/raw/Activity_expenditure/expenditure_all.csv",   # <-- set real name
)

# accounting (18 cols) — the flattened voucher file
df_ac = pd.read_csv(
    directories.PROCESSED_DATA / "all_vouchers_flat.csv",   # path from config
)

print("loaded:", df_pl.shape, df_ax.shape, df_ac.shape)     # expect (12704,87) (12730,25) (12440,18)

loaded: (12704, 87) (12730, 25) (12440, 18)


In [5]:
# ============================================================
# BLOCK 2 — clean & standardize
# ============================================================
import pandas as pd

def to_code(series):
    # any int/float/text code -> clean string ('119598', never 119598.0)
    s = series.astype("string").str.strip()
    return s.str.replace(r"\.0$", "", regex=True)

def to_fiscal_year(series):
    # int 2020 -> "2020-2021"; "2020-2021" stays as-is
    def conv(v):
        if pd.isna(v):
            return pd.NA
        s = str(v).strip()
        if "-" in s:
            return s
        return f"{int(float(s))}-{int(float(s)) + 1}"
    return series.map(conv).astype("string")

# ---------- PLANNING ----------
pl = df_pl.copy()
for c in ["gp_lgd_code", "plan_code", "activity_code"]:      # three join keys
    pl[c] = to_code(pl[c])
pl["fiscal_year"] = to_fiscal_year(pl["plan_year"])

# ---------- EXPENDITURE ----------
ax = df_ax.copy().rename(columns={                           # camelCase/spaced -> snake_case
    "planYear": "plan_year", "stateName": "state_name", "zpName": "zp_name",
    "blockName": "block_name", "gpName": "gp_name", "gpCode": "gp_lgd_code",
    "planType": "plan_type", "approvalDate": "approval_date", "planCode": "plan_code",
    "S.No.": "s_no", "Activity Code": "activity_code", "Activity Name": "activity_name",
    "Activity For": "activity_for", "Focus Area": "focus_area",
    "Approved Cost in Action Plan": "approved_cost_action_plan",
    "Technical Approved Cost": "technical_approved_cost",
    "Admin Approved Cost": "admin_approved_cost", "Scheme Name": "scheme_name",
    "General": "general", "SC": "sc", "ST": "st",
    "Total Expenditure": "total_expenditure",
    "Voucher Date": "voucher_date_list", "Voucher No": "voucher_no_list",   # _list = still pipe-delimited
    "Voucher Cost": "voucher_cost_list",
})
for c in ["gp_lgd_code", "plan_code", "activity_code"]:
    ax[c] = to_code(ax[c])
ax["fiscal_year"] = to_fiscal_year(ax["plan_year"])
ax["approval_date"] = pd.to_datetime(ax["approval_date"], errors="coerce")   # ISO -> datetime

# ---------- ACCOUNTING ----------
ac = df_ac.copy()
for c in ["gp_lgd_code", "state", "district", "block", "voucher_id", "voucher_no"]:  # keys + geo codes
    ac[c] = to_code(ac[c])
ac["date"] = pd.to_datetime(ac["date"], dayfirst=True, errors="coerce")      # DD/MM/YYYY -> datetime

# ---------- verify ----------
print("shapes:  pl", pl.shape, " ax", ax.shape, " ac", ac.shape)
print("keys:", pl["activity_code"].dtype, ax["activity_code"].dtype, ac["voucher_no"].dtype)

shapes:  pl (12704, 88)  ax (12730, 26)  ac (12440, 18)
keys: string string string


In [6]:
import pandas as pd

# which tables actually exist?
tables = pd.read_sql("SHOW TABLES", engine)["name"].tolist()
print("TABLES (", len(tables), "):")
print(tables)

# exact columns of each, so Block 4 matches reality
print("\n--- columns per table ---")
for t in tables:
    cols = pd.read_sql(f"PRAGMA table_info('{t}')", engine)["name"].tolist()
    print(f"\n{t}  ({len(cols)}):")
    print(" ", cols)

TABLES ( 12 ):
['activity_asset', 'activity_community_service', 'activity_delegation', 'activity_expenditure', 'activity_fund', 'activity_nsap', 'activity_training', 'activity_voucher', 'gram_panchayat', 'plan', 'planned_activity', 'voucher']

--- columns per table ---

activity_asset  (21):
  ['activity_code', 'main_asset_category', 'main_asset_subcategory', 'main_asset_unit_type', 'main_asset_unit_count', 'asset_type', 'asset_category', 'asset_subcategory', 'asset_coverage_code', 'asset_name', 'asset_unit_type', 'asset_unit_count', 'asset_unit_cost', 'asset_parameter_type', 'asset_details_raw', 'asset_loc_code', 'asset_loc_unit_code', 'asset_loc_unit_type', 'asset_loc_unit_count', 'asset_loc_unit_cost_total', 'asset_loc_overflow_json']

activity_community_service  (5):
  ['activity_code', 'community_service_raw', 'community_service_code', 'community_service_duration', 'community_beneficiaries_expected']

activity_delegation  (8):
  ['activity_code', 'is_delegated', 'delegated_unit_co

In [24]:
# ============================================================
# BLOCK 3 — create all 11 tables with keys, final names
# ============================================================
from sqlalchemy import text
import pandas as pd

# ---- 1. full reset (children -> parents) ----
all_tables = [
    "activity_voucher", "voucher", "activity_expenditure", "activity_nsap",
    "activity_community_service", "activity_training", "activity_fund",
    "activity_asset", "activity_delegation", "planned_activity", "plan", "gram_panchayat",
]
with engine.begin() as conn:
    for t in all_tables:
        conn.execute(text(f"DROP TABLE IF EXISTS {t}"))
        conn.execute(text(f"DROP TABLE IF EXISTS {t}_new"))

# ---- 2. DDL: parents before children ----
ddl = """
CREATE TABLE gram_panchayat (
    gp_lgd_code   VARCHAR PRIMARY KEY,
    gp_name VARCHAR, state_code VARCHAR, state_name VARCHAR,
    district_code VARCHAR, zp_name VARCHAR, block_code VARCHAR, block_name VARCHAR
);

CREATE TABLE plan (
    plan_code VARCHAR PRIMARY KEY, gp_lgd_code VARCHAR,
    fiscal_year VARCHAR, plan_type VARCHAR, approval_date TIMESTAMP, plan_code_status VARCHAR,
    FOREIGN KEY (gp_lgd_code) REFERENCES gram_panchayat (gp_lgd_code)
);

CREATE TABLE planned_activity (
    activity_code VARCHAR PRIMARY KEY, plan_code VARCHAR, gp_lgd_code VARCHAR,
    fiscal_year VARCHAR, source_file VARCHAR, activity_type BIGINT, activity_name VARCHAR,
    activity_desc VARCHAR, focus_area BIGINT, activity_for BIGINT, work_type BIGINT,
    is_costless_activity BIGINT, total_cost DOUBLE, operation_type DOUBLE,
    operation_remarks VARCHAR, output_type BIGINT, activity_status BIGINT,
    FOREIGN KEY (plan_code)   REFERENCES plan (plan_code),
    FOREIGN KEY (gp_lgd_code) REFERENCES gram_panchayat (gp_lgd_code)
);

CREATE TABLE activity_delegation (
    activity_code VARCHAR PRIMARY KEY,
    is_delegated DOUBLE, delegated_unit_code DOUBLE, delegated_unit_type DOUBLE,
    delegated_unit_level DOUBLE, delegated_unit_category DOUBLE, is_shareable BOOLEAN,
    delegated_parent_unit_code DOUBLE,
    FOREIGN KEY (activity_code) REFERENCES planned_activity (activity_code)
);

CREATE TABLE activity_asset (
    activity_code VARCHAR PRIMARY KEY,
    main_asset_category DOUBLE, main_asset_subcategory DOUBLE, main_asset_unit_type DOUBLE,
    main_asset_unit_count DOUBLE, asset_type DOUBLE, asset_category DOUBLE,
    asset_subcategory DOUBLE, asset_coverage_code VARCHAR, asset_name DOUBLE,
    asset_unit_type DOUBLE, asset_unit_count DOUBLE, asset_unit_cost DOUBLE,
    asset_parameter_type DOUBLE, asset_details_raw DOUBLE, asset_loc_code DOUBLE,
    asset_loc_unit_code DOUBLE, asset_loc_unit_type DOUBLE, asset_loc_unit_count DOUBLE,
    asset_loc_unit_cost_total DOUBLE, asset_loc_overflow_json VARCHAR,
    FOREIGN KEY (activity_code) REFERENCES planned_activity (activity_code)
);

CREATE TABLE activity_fund (
    activity_code VARCHAR PRIMARY KEY,
    fund_scheme_code DOUBLE, fund_component_code DOUBLE, fund_tied_general DOUBLE,
    fund_tied_sc DOUBLE, fund_tied_st DOUBLE, fund_untied_general DOUBLE,
    fund_untied_sc DOUBLE, fund_untied_st DOUBLE, fund_amount_total DOUBLE,
    fund_tied_abandoned_general DOUBLE, fund_tied_abandoned_sc DOUBLE,
    fund_tied_abandoned_st DOUBLE, fund_untied_abandoned_general DOUBLE,
    fund_untied_abandoned_sc DOUBLE, fund_untied_abandoned_st DOUBLE,
    fund_overflow_json VARCHAR,
    FOREIGN KEY (activity_code) REFERENCES planned_activity (activity_code)
);

CREATE TABLE activity_training (
    activity_code VARCHAR PRIMARY KEY,
    training_capacity_raw DOUBLE, training_category_code DOUBLE, training_organiser_code DOUBLE,
    training_subject VARCHAR, training_trainees_total DOUBLE, training_duration_days DOUBLE,
    FOREIGN KEY (activity_code) REFERENCES planned_activity (activity_code)
);

CREATE TABLE activity_community_service (
    activity_code VARCHAR PRIMARY KEY,
    community_service_raw DOUBLE, community_service_code DOUBLE,
    community_service_duration DOUBLE, community_beneficiaries_expected DOUBLE,
    FOREIGN KEY (activity_code) REFERENCES planned_activity (activity_code)
);

CREATE TABLE activity_nsap (
    nsap_id INTEGER PRIMARY KEY, activity_code VARCHAR,
    category VARCHAR, age_band VARCHAR, gender VARCHAR, beneficiary_count DECIMAL(16,2),
    FOREIGN KEY (activity_code) REFERENCES planned_activity (activity_code)
);

CREATE TABLE activity_expenditure (
    expenditure_id INTEGER PRIMARY KEY, activity_code VARCHAR, plan_code VARCHAR,
    gp_lgd_code VARCHAR, fiscal_year VARCHAR, s_no BIGINT, scheme_name VARCHAR,
    approved_cost_action_plan DECIMAL(16,2), technical_approved_cost DECIMAL(16,2),
    admin_approved_cost DECIMAL(16,2), general DECIMAL(16,2), sc DECIMAL(16,2),
    st DECIMAL(16,2), total_expenditure DECIMAL(16,2),
    FOREIGN KEY (plan_code)   REFERENCES plan (plan_code),
    FOREIGN KEY (gp_lgd_code) REFERENCES gram_panchayat (gp_lgd_code)
);

CREATE TABLE voucher (
    voucher_pk  INTEGER PRIMARY KEY,
    gp_lgd_code VARCHAR, fiscal_year VARCHAR, voucher_no VARCHAR,
    voucher_id  VARCHAR,
    direction VARCHAR, type VARCHAR, date DATE, month VARCHAR,
    amount DECIMAL(16,2),
    UNIQUE (gp_lgd_code, fiscal_year, voucher_no),
    FOREIGN KEY (gp_lgd_code) REFERENCES gram_panchayat (gp_lgd_code)
);

CREATE TABLE activity_voucher (
    expenditure_id INTEGER,
    voucher_pk     INTEGER,
    gp_lgd_code VARCHAR, fiscal_year VARCHAR, voucher_no VARCHAR,
    voucher_date DATE, voucher_cost DECIMAL(16,2),
    FOREIGN KEY (expenditure_id) REFERENCES activity_expenditure (expenditure_id)
);
"""

# ---- 3. execute with per-statement reporting ----
statements = [s for s in ddl.split(";") if s.strip()]
with engine.begin() as conn:
    for i, stmt in enumerate(statements):
        try:
            conn.execute(text(stmt))
            name = stmt.strip().split("(")[0].replace("CREATE TABLE", "").strip()
            print(f"OK  #{i:2}: {name}")
        except Exception as e:
            print(f"FAIL #{i:2}: {stmt.strip()[:60]}")
            print("     ", repr(e)); raise

print("\nTABLES NOW:", pd.read_sql("SHOW TABLES", engine)["name"].tolist())

OK  # 0: gram_panchayat
OK  # 1: plan
OK  # 2: planned_activity
OK  # 3: activity_delegation
OK  # 4: activity_asset
OK  # 5: activity_fund
OK  # 6: activity_training
OK  # 7: activity_community_service
OK  # 8: activity_nsap
OK  # 9: activity_expenditure
OK  #10: voucher
OK  #11: activity_voucher

TABLES NOW: ['activity_asset', 'activity_community_service', 'activity_delegation', 'activity_expenditure', 'activity_fund', 'activity_nsap', 'activity_training', 'activity_voucher', 'gram_panchayat', 'plan', 'planned_activity', 'voucher']


In [8]:
print(engine.url)   # must end in panchayat_1.duckdb

duckdb:////home/admin_abhi/GitHub/ai-for-panchayats/data/interim/panchayat_1.duckdb


In [25]:
# ============================================================
# BLOCK 4 — load all 11 tables (idempotent)
# ============================================================
import re
import pandas as pd
from sqlalchemy import text

# ---- 0. clear (children -> parents) so re-runs don't double ----
clear_order = [
    "activity_voucher", "voucher", "activity_expenditure", "activity_nsap",
    "activity_community_service", "activity_training", "activity_fund",
    "activity_asset", "activity_delegation", "planned_activity", "plan", "gram_panchayat",
]
with engine.begin() as conn:
    for t in clear_order:
        conn.execute(text(f"DELETE FROM {t}"))

# ---------- DIMENSIONS ----------
gp = (ac[["gp_lgd_code", "gp_name", "state", "district", "block"]]
      .rename(columns={"state": "state_code", "district": "district_code", "block": "block_code"})
      .merge(ax[["gp_lgd_code", "state_name", "zp_name", "block_name"]],
             on="gp_lgd_code", how="outer")
      .drop_duplicates("gp_lgd_code")
      [["gp_lgd_code", "gp_name", "state_code", "state_name",
        "district_code", "zp_name", "block_code", "block_name"]])
gp.to_sql("gram_panchayat", engine, if_exists="append", index=False)

plan_df = ax[["plan_code", "gp_lgd_code", "fiscal_year", "plan_type", "approval_date"]].copy()
plan_df["plan_code_status"] = pd.NA
plan_df = plan_df.dropna(subset=["plan_code"]).drop_duplicates("plan_code")
plan_df.to_sql("plan", engine, if_exists="append", index=False)

# ---------- PLANNING CORE + SATELLITES ----------
core_cols = ["activity_code", "plan_code", "gp_lgd_code", "fiscal_year", "source_file",
             "activity_type", "activity_name", "activity_desc", "focus_area", "activity_for",
             "work_type", "is_costless_activity", "total_cost", "operation_type",
             "operation_remarks", "output_type", "activity_status"]
pl[core_cols].drop_duplicates("activity_code").to_sql(
    "planned_activity", engine, if_exists="append", index=False)

sat_groups = {
    "activity_delegation": ["is_delegated", "delegated_unit_code", "delegated_unit_type",
        "delegated_unit_level", "delegated_unit_category", "is_shareable", "delegated_parent_unit_code"],
    "activity_asset": ["main_asset_category", "main_asset_subcategory", "main_asset_unit_type",
        "main_asset_unit_count", "asset_type", "asset_category", "asset_subcategory",
        "asset_coverage_code", "asset_name", "asset_unit_type", "asset_unit_count", "asset_unit_cost",
        "asset_parameter_type", "asset_details_raw", "asset_loc_code", "asset_loc_unit_code",
        "asset_loc_unit_type", "asset_loc_unit_count", "asset_loc_unit_cost_total", "asset_loc_overflow_json"],
    "activity_fund": ["fund_scheme_code", "fund_component_code", "fund_tied_general", "fund_tied_sc",
        "fund_tied_st", "fund_untied_general", "fund_untied_sc", "fund_untied_st", "fund_amount_total",
        "fund_tied_abandoned_general", "fund_tied_abandoned_sc", "fund_tied_abandoned_st",
        "fund_untied_abandoned_general", "fund_untied_abandoned_sc", "fund_untied_abandoned_st", "fund_overflow_json"],
    "activity_training": ["training_capacity_raw", "training_category_code", "training_organiser_code",
        "training_subject", "training_trainees_total", "training_duration_days"],
    "activity_community_service": ["community_service_raw", "community_service_code",
        "community_service_duration", "community_beneficiaries_expected"],
}
for t, cols in sat_groups.items():
    pl[["activity_code"] + cols].drop_duplicates("activity_code").to_sql(
        t, engine, if_exists="append", index=False)

# ---------- NSAP (all-null in this dataset -> 0 rows, which is correct) ----------
nsap_map = {
    "nsap_old_age_lt80_male": ("old_age","lt80","male"), "nsap_old_age_lt80_female": ("old_age","lt80","female"),
    "nsap_old_age_lt80_transgender": ("old_age","lt80","transgender"),
    "nsap_old_age_ge80_male": ("old_age","ge80","male"), "nsap_old_age_ge80_female": ("old_age","ge80","female"),
    "nsap_old_age_ge80_transgender": ("old_age","ge80","transgender"),
    "nsap_disabled_male": ("disabled","na","male"), "nsap_disabled_female": ("disabled","na","female"),
    "nsap_disabled_transgender": ("disabled","na","transgender"),
    "nsap_widow_male": ("widow","na","male"), "nsap_widow_female": ("widow","na","female"),
    "nsap_widow_transgender": ("widow","na","transgender"),
}
nsap = pl[["activity_code"] + list(nsap_map)].melt(
    id_vars="activity_code", var_name="col", value_name="beneficiary_count")
nsap["category"] = nsap["col"].map(lambda c: nsap_map[c][0])
nsap["age_band"] = nsap["col"].map(lambda c: nsap_map[c][1])
nsap["gender"]   = nsap["col"].map(lambda c: nsap_map[c][2])
nsap = nsap[nsap["beneficiary_count"].notna() & (nsap["beneficiary_count"] != 0)]
nsap = nsap[["activity_code", "category", "age_band", "gender", "beneficiary_count"]].reset_index(drop=True)
nsap.insert(0, "nsap_id", range(1, len(nsap) + 1))
if len(nsap):
    nsap.to_sql("activity_nsap", engine, if_exists="append", index=False)

# ---------- EXPENDITURE (expenditure_id assigned here; bridge reuses it) ----------
exp = ax[["activity_code", "plan_code", "gp_lgd_code", "fiscal_year", "s_no", "scheme_name",
          "approved_cost_action_plan", "technical_approved_cost", "admin_approved_cost",
          "general", "sc", "st", "total_expenditure"]].copy().reset_index(drop=True)
exp.insert(0, "expenditure_id", range(1, len(exp) + 1))
exp.to_sql("activity_expenditure", engine, if_exists="append", index=False)

# ---------- VOUCHER (all rows kept; voucher_pk = row number) ----------
vouch = ac[["gp_lgd_code", "fiscal_year", "voucher_no", "voucher_id",
            "direction", "type", "date", "month", "amount"]].copy().reset_index(drop=True)
vouch.insert(0, "voucher_pk", range(1, len(vouch) + 1))
vouch.to_sql("voucher", engine, if_exists="append", index=False)

# ---------- BRIDGE (year parsed from voucher_no, then look up voucher_pk) ----------
def year_from_vno(vno):
    """XVFC/2025-26/P/143 -> '2025-2026' (the voucher's own year, not the plan's)."""
    m = re.search(r"/(\d{4})-(\d{2})/", str(vno))
    if not m:
        return pd.NA
    start = int(m.group(1))
    return f"{start}-{start + 1}"

br = ax[["voucher_no_list", "voucher_date_list", "voucher_cost_list", "gp_lgd_code"]].copy()
br["expenditure_id"] = exp["expenditure_id"].values      # align to the same ax rows
br = br.dropna(subset=["voucher_no_list"])

rows = []
for _, r in br.iterrows():
    nos   = str(r["voucher_no_list"]).split(" | ")
    dates = str(r["voucher_date_list"]).split(" | ")
    costs = str(r["voucher_cost_list"]).split(" | ")
    for i, vno in enumerate(nos):
        vno = vno.strip()
        d = dates[i] if i < len(dates) else None
        c = costs[i] if i < len(costs) else None
        rows.append({
            "expenditure_id": r["expenditure_id"],
            "gp_lgd_code": r["gp_lgd_code"],
            "fiscal_year": year_from_vno(vno),           # voucher's own year
            "voucher_no": vno,
            "voucher_date": pd.to_datetime(d, dayfirst=True, errors="coerce") if d else None,
            "voucher_cost": pd.to_numeric(c, errors="coerce") if c else None,
        })
bridge = pd.DataFrame(rows)

bridge = bridge.merge(                                    # attach voucher_pk
    vouch[["voucher_pk", "gp_lgd_code", "fiscal_year", "voucher_no"]],
    on=["gp_lgd_code", "fiscal_year", "voucher_no"], how="left")

bridge = bridge[["expenditure_id", "voucher_pk", "gp_lgd_code", "fiscal_year",
                 "voucher_no", "voucher_date", "voucher_cost"]]
bridge.to_sql("activity_voucher", engine, if_exists="append", index=False)

matched = bridge["voucher_pk"].notna().sum()
print(f"bridge rows: {len(bridge)}  matched: {matched}  unmatched: {len(bridge)-matched}"
      f"  ({matched/len(bridge)*100:.1f}%)\n")

# ---------- VERIFY ----------
for t in ["gram_panchayat", "plan", "planned_activity", "activity_delegation",
          "activity_asset", "activity_fund", "activity_training",
          "activity_community_service", "activity_nsap", "activity_expenditure",
          "voucher", "activity_voucher"]:
    n = pd.read_sql(f"SELECT count(*) FROM {t}", engine).iloc[0, 0]
    print(f"{t:28} {n}")

bridge rows: 5976  matched: 5488  unmatched: 488  (91.8%)

gram_panchayat               20
plan                         204
planned_activity             12704
activity_delegation          12704
activity_asset               12704
activity_fund                12704
activity_training            12704
activity_community_service   12704
activity_nsap                0
activity_expenditure         12730
voucher                      12440
activity_voucher             5976


In [34]:
import pandas as pd

# what's distinctive about the unmatched bridge rows?
un = pd.read_sql("""
    SELECT gp_lgd_code, fiscal_year, count(*) AS n
    FROM activity_voucher
    WHERE voucher_pk IS NULL
    GROUP BY gp_lgd_code, fiscal_year
    ORDER BY n DESC
    LIMIT 15
""", engine)
print("unmatched, by GP and year:")
print(un.to_string(index=False))

# does the accounting file even cover those GP-years?
print("\nGP-years present in voucher table:")
print(pd.read_sql("""
    SELECT fiscal_year, count(DISTINCT gp_lgd_code) AS gps, count(*) AS vouchers
    FROM voucher GROUP BY fiscal_year ORDER BY fiscal_year
""", engine).to_string(index=False))

unmatched, by GP and year:
gp_lgd_code fiscal_year  n
     119862   2026-2027 82
     119717   2026-2027 74
     118939   2026-2027 44
     275075   2026-2027 40
     116350   2026-2027 32
     117835   2026-2027 27
     119599   2026-2027 26
     121162   2026-2027 24
     117153   2026-2027 23
     117951   2026-2027 23
     118012   2026-2027 21
     116400   2026-2027 21
     116397   2026-2027 19
     116936   2026-2027 16
     119605   2026-2027  8

GP-years present in voucher table:
fiscal_year  gps  vouchers
  2020-2021   20      2234
  2021-2022   20      2094
  2022-2023   20      1988
  2023-2024   20      1576
  2024-2025   20      1998
  2025-2026   20      2550


In [17]:
import pandas as pd
# columns I declared DOUBLE but that might actually hold JSON/text
suspects = ["fund_overflow_json", "asset_loc_overflow_json", "asset_details_raw",
            "training_capacity_raw", "community_service_raw"]
for c in suspects:
    if c in pl.columns:
        # show dtype + a non-null sample
        non_null = pl[c].dropna()
        sample = non_null.iloc[0] if len(non_null) else "(all null)"
        print(f"{c:26} dtype={str(pl[c].dtype):10} sample={str(sample)[:60]}")

fund_overflow_json         dtype=str        sample=[{"schemeCode": 1778, "componentCode": 4250, "tiedAmountGen"
asset_loc_overflow_json    dtype=str        sample=[{"astLocCd": 380454, "astPlnUntCd": null, "astPlnUntTyp": n
asset_details_raw          dtype=float64    sample=(all null)
training_capacity_raw      dtype=float64    sample=(all null)
community_service_raw      dtype=float64    sample=(all null)


In [26]:
import pandas as pd

# 1. NSAP — are the source columns actually all empty?
nsap_cols = [c for c in pl.columns if c.startswith("nsap_") or c == "pmayg_raw"]
print("NSAP source columns — non-null counts:")
print(pl[nsap_cols].notna().sum().to_string())
print("non-zero values total:", (pl[nsap_cols].fillna(0) != 0).sum().sum())

# 2. VOUCHER — were the 117 dropped rows真 duplicates?
print("\nac rows:", len(ac), " distinct voucher_id:", ac["voucher_id"].nunique())
dupes = ac[ac.duplicated("voucher_id", keep=False)].sort_values("voucher_id")
print("rows sharing a voucher_id:", len(dupes))
if len(dupes):
    print(dupes[["voucher_id","voucher_no","gp_lgd_code","fiscal_year","amount"]].head(6).to_string(index=False))

# 3. BRIDGE — expected voucher count from the source lists
expected = ax["voucher_no_list"].dropna().apply(lambda s: len(str(s).split(" | "))).sum()
print("\nbridge rows expected from ax:", expected, " loaded:", 
      pd.read_sql("SELECT count(*) FROM activity_voucher", engine).iloc[0,0])

NSAP source columns — non-null counts:
pmayg_raw                        0
nsap_raw                         0
nsap_old_age_lt80_male           0
nsap_old_age_lt80_female         0
nsap_old_age_lt80_transgender    0
nsap_old_age_ge80_male           0
nsap_old_age_ge80_female         0
nsap_old_age_ge80_transgender    0
nsap_disabled_male               0
nsap_disabled_female             0
nsap_disabled_transgender        0
nsap_widow_male                  0
nsap_widow_female                0
nsap_widow_transgender           0
non-zero values total: 0

ac rows: 12440  distinct voucher_id: 12323
rows sharing a voucher_id: 233
voucher_id          voucher_no gp_lgd_code fiscal_year   amount
    100716 MGNREGA/2022-23/P/5      116438   2022-2023  27972.0
    100716     FFC/2023-24/P/9      275075   2023-2024 252450.0
     11587    XVFC/2021-22/P/1      116438   2021-2022  96932.0
     11587    XVFC/2022-23/P/2      121526   2022-2023 396649.0
     11754     PDS/2021-22/R/3      119598   2021-2

In [32]:
#updated check 
import pandas as pd

# 1. do the keys actually exist now?
print("--- constraints ---")
print(pd.read_sql("""
    SELECT table_name, constraint_type
    FROM information_schema.table_constraints
    ORDER BY table_name, constraint_type
""", engine).to_string(index=False))

# 2. row counts
print("\n--- row counts ---")
for t in ["gram_panchayat","plan","planned_activity","activity_delegation","activity_asset",
          "activity_fund","activity_training","activity_community_service","activity_nsap",
          "activity_expenditure","voucher","activity_voucher"]:
    n = pd.read_sql(f"SELECT count(*) FROM {t}", engine).iloc[0,0]
    print(f"{t:28} {n}")

# 3. money check: does the DB total match the source frame?
db_total  = pd.read_sql("SELECT sum(amount) FROM voucher", engine).iloc[0,0]
src_total = ac["amount"].sum()
print(f"\nvoucher amount — db: {db_total:,.2f}  source: {src_total:,.2f}  diff: {float(db_total)-src_total:,.2f}")

# 4. how well does the bridge link to voucher?
print("\n--- bridge linkage ---")
print(pd.read_sql("""
    SELECT
      count(*)                                  AS bridge_rows,
      count(voucher_pk)                          AS matched,
      count(*) - count(voucher_pk)               AS unmatched
    FROM activity_voucher
""", engine).to_string(index=False))

--- constraints ---
                table_name constraint_type
            activity_asset           CHECK
            activity_asset     FOREIGN KEY
            activity_asset     PRIMARY KEY
activity_community_service           CHECK
activity_community_service     FOREIGN KEY
activity_community_service     PRIMARY KEY
       activity_delegation           CHECK
       activity_delegation     FOREIGN KEY
       activity_delegation     PRIMARY KEY
      activity_expenditure           CHECK
      activity_expenditure     FOREIGN KEY
      activity_expenditure     FOREIGN KEY
      activity_expenditure     PRIMARY KEY
             activity_fund           CHECK
             activity_fund     FOREIGN KEY
             activity_fund     PRIMARY KEY
             activity_nsap           CHECK
             activity_nsap     FOREIGN KEY
             activity_nsap     PRIMARY KEY
         activity_training           CHECK
         activity_training     FOREIGN KEY
         activity_training     PRI

In [33]:
import re
import pandas as pd

# extract the year embedded in voucher_no, e.g. "XVFC/2025-26/P/143" -> "2025-2026"
def year_from_vno(vno):
    m = re.search(r"/(\d{4})-(\d{2})/", str(vno))
    if not m:
        return pd.NA
    start = int(m.group(1))
    return f"{start}-{start+1}"

sample = ac["voucher_no"].dropna().head(5)
print("parse check on ac.voucher_no:")
for v in sample:
    print(f"  {v:28} -> {year_from_vno(v)}")

# how many voucher numbers parse successfully?
parsed = ac["voucher_no"].map(year_from_vno)
print(f"\nparsed: {parsed.notna().sum()} of {len(ac)}")

# does the parsed year agree with ac's own fiscal_year?
agree = (parsed == ac["fiscal_year"]).sum()
print(f"parsed year matches ac.fiscal_year: {agree} of {parsed.notna().sum()}")

parse check on ac.voucher_no:
  XVFC/2025-26/R/1             -> 2025-2026
  XVFC/2025-26/R/2             -> 2025-2026
  XVFC/2025-26/R/3             -> 2025-2026
  XVFC/2025-26/R/4             -> 2025-2026
  5THSFC/2025-26/R/1           -> 2025-2026

parsed: 12440 of 12440
parsed year matches ac.fiscal_year: 12440 of 12440


In [35]:
pd.read_sql("""
    SELECT g.gp_name, a.fiscal_year,
           count(*) AS activities,
           sum(e.total_expenditure) AS spent
    FROM planned_activity a
    JOIN gram_panchayat g USING (gp_lgd_code)
    LEFT JOIN activity_expenditure e USING (activity_code)
    GROUP BY 1, 2 ORDER BY 1, 2
""", engine)

,gp_name,fiscal_year,activities,spent
0,Andhrua,2020-2021,16,3432694.00
1,Andhrua,2021-2022,27,1661841.00
2,Andhrua,2022-2023,35,3107156.00
3,Andhrua,2023-2024,211,2401719.81
4,Andhrua,2024-2025,118,2649400.00
...,...,...,...,...
115,Sharagada,2021-2022,30,2222728.00
116,Sharagada,2022-2023,29,2935902.00
117,Sharagada,2023-2024,433,4441030.00
118,Sharagada,2024-2025,276,3493575.00


In [39]:
import pandas as pd

pd.read_sql("""
    SELECT p.plan_type,
           count(*) AS activities
    FROM planned_activity a
    JOIN gram_panchayat g USING (gp_lgd_code)
    JOIN plan p USING (plan_code)
    WHERE g.gp_lgd_code = '119598'          -- Andhrua
      AND a.fiscal_year = '2025-2026'
    GROUP BY p.plan_type
    ORDER BY p.plan_type
""", engine)

,plan_type,activities
0,Main,84
1,Supplementary,2


In [41]:
# total activity wise expenditure for andhrua in 2025-26
import pandas as pd

pd.read_sql("""
    SELECT sum(e.total_expenditure) AS total_expenditure
    FROM activity_expenditure e
    JOIN gram_panchayat g USING (gp_lgd_code)
    WHERE g.gp_lgd_code = '119598'          -- Andhrua
      AND e.fiscal_year = '2025-2026'
""", engine)

,total_expenditure
0,1601731.0


In [ ]:
# overall expenditure in 2025-26
pd.read_sql("""
    SELECT sum(amount) AS total_payments
    FROM voucher
    WHERE gp_lgd_code = '119598'
      AND fiscal_year = '2025-2026'
      AND direction = 'payment'
""", engine)

,total_payments
0,9471754.81


In [43]:
pd.read_sql("""
    SELECT direction,
           count(*)     AS vouchers,
           sum(amount)  AS total
    FROM voucher
    WHERE gp_lgd_code = '119598'
      AND fiscal_year = '2025-2026'
    GROUP BY direction
""", engine)

,direction,vouchers,total
0,receipt,22,5283358.00
1,payment,264,9471754.81


In [44]:
import pandas as pd

pd.read_sql("""
    SELECT a.fiscal_year,
           count(DISTINCT a.activity_code)   AS activities,
           sum(a.total_cost)                 AS planned_cost,
           sum(e.total_expenditure)          AS actual_expenditure,
           round(100.0 * sum(e.total_expenditure)
                 / nullif(sum(a.total_cost), 0), 1) AS utilisation_pct
    FROM planned_activity a
    JOIN gram_panchayat g USING (gp_lgd_code)
    LEFT JOIN activity_expenditure e USING (activity_code)
    WHERE g.gp_lgd_code = '119598'          -- Andhrua
    GROUP BY a.fiscal_year
    ORDER BY a.fiscal_year
""", engine)

,fiscal_year,activities,planned_cost,actual_expenditure,utilisation_pct
0,2020-2021,16,3947654.0,3432694.00,87.0
1,2021-2022,27,3636707.0,1661841.00,45.7
2,2022-2023,35,6000000.0,3107156.00,51.8
3,2023-2024,211,4069134.0,2401719.81,59.0
4,2024-2025,118,3518192.0,2649400.00,75.3
5,2025-2026,86,3888192.0,1601731.00,41.2


In [45]:
# ============================================================
# BLOCK 5 — validate (counts, keys, orphans, reconciliation)
# ============================================================
import pandas as pd

def q(sql):
    return pd.read_sql(sql, engine)

print("=" * 66)
print("1. ROW COUNTS  (db vs expected from source frames)")
print("=" * 66)
expected = {
    "gram_panchayat": ac["gp_lgd_code"].nunique(),
    "plan": ax["plan_code"].dropna().nunique(),
    "planned_activity": pl["activity_code"].nunique(),
    "activity_delegation": pl["activity_code"].nunique(),
    "activity_asset": pl["activity_code"].nunique(),
    "activity_fund": pl["activity_code"].nunique(),
    "activity_training": pl["activity_code"].nunique(),
    "activity_community_service": pl["activity_code"].nunique(),
    "activity_expenditure": len(ax),
    "voucher": len(ac),
}
for t, exp_n in expected.items():
    n = q(f"SELECT count(*) FROM {t}").iloc[0, 0]
    flag = "OK" if n == exp_n else f"MISMATCH (expected {exp_n})"
    print(f"  {t:28} {n:>7}   {flag}")
for t in ["activity_nsap", "activity_voucher"]:
    n = q(f"SELECT count(*) FROM {t}").iloc[0, 0]
    print(f"  {t:28} {n:>7}   (derived)")

print("\n" + "=" * 66)
print("2. KEY CONSTRAINTS")
print("=" * 66)
cons = q("""
    SELECT table_name, constraint_type
    FROM information_schema.table_constraints
    WHERE constraint_type IN ('PRIMARY KEY','FOREIGN KEY','UNIQUE')
    ORDER BY table_name, constraint_type
""")
print(cons.to_string(index=False))
print(f"\n  total constraints: {len(cons)}")

print("\n" + "=" * 66)
print("3. DUPLICATE PRIMARY KEYS  (should all be 0)")
print("=" * 66)
pk_checks = [
    ("gram_panchayat", "gp_lgd_code"), ("plan", "plan_code"),
    ("planned_activity", "activity_code"), ("activity_delegation", "activity_code"),
    ("activity_asset", "activity_code"), ("activity_fund", "activity_code"),
    ("activity_training", "activity_code"), ("activity_community_service", "activity_code"),
    ("activity_expenditure", "expenditure_id"), ("voucher", "voucher_pk"),
]
for t, k in pk_checks:
    d = q(f"SELECT count(*) - count(DISTINCT {k}) FROM {t}").iloc[0, 0]
    print(f"  {t:28} {k:16} {d}   {'OK' if d == 0 else 'DUPLICATES'}")

print("\n" + "=" * 66)
print("4. ORPHAN FOREIGN KEYS  (child key missing from parent)")
print("=" * 66)
orphans = {
    "plan -> gram_panchayat":
        "SELECT count(*) FROM plan c LEFT JOIN gram_panchayat p USING(gp_lgd_code) WHERE p.gp_lgd_code IS NULL",
    "planned_activity -> plan":
        "SELECT count(*) FROM planned_activity c LEFT JOIN plan p USING(plan_code) WHERE p.plan_code IS NULL",
    "planned_activity -> gram_panchayat":
        "SELECT count(*) FROM planned_activity c LEFT JOIN gram_panchayat p USING(gp_lgd_code) WHERE p.gp_lgd_code IS NULL",
    "activity_expenditure -> planned_activity (unenforced)":
        "SELECT count(*) FROM activity_expenditure c LEFT JOIN planned_activity p USING(activity_code) WHERE p.activity_code IS NULL",
    "voucher -> gram_panchayat":
        "SELECT count(*) FROM voucher c LEFT JOIN gram_panchayat p USING(gp_lgd_code) WHERE p.gp_lgd_code IS NULL",
    "activity_voucher -> activity_expenditure":
        "SELECT count(*) FROM activity_voucher c LEFT JOIN activity_expenditure p USING(expenditure_id) WHERE p.expenditure_id IS NULL",
}
for sat in ["activity_delegation", "activity_asset", "activity_fund",
            "activity_training", "activity_community_service"]:
    orphans[f"{sat} -> planned_activity"] = (
        f"SELECT count(*) FROM {sat} c LEFT JOIN planned_activity p USING(activity_code) "
        f"WHERE p.activity_code IS NULL")
for label, sql in orphans.items():
    n = q(sql).iloc[0, 0]
    print(f"  {label:52} {n}   {'OK' if n == 0 else 'ORPHANS'}")

print("\n" + "=" * 66)
print("5. MONEY RECONCILIATION  (db vs source frames)")
print("=" * 66)
checks = [
    ("voucher.amount", q("SELECT sum(amount) FROM voucher").iloc[0, 0], ac["amount"].sum()),
    ("expenditure.total_expenditure",
     q("SELECT sum(total_expenditure) FROM activity_expenditure").iloc[0, 0],
     ax["total_expenditure"].sum()),
    ("planned_activity.total_cost",
     q("SELECT sum(total_cost) FROM planned_activity").iloc[0, 0],
     pl.drop_duplicates("activity_code")["total_cost"].sum()),
]
for label, db_v, src_v in checks:
    db_v = float(db_v or 0); src_v = float(src_v or 0)
    diff = db_v - src_v
    print(f"  {label:32} db={db_v:>18,.2f}  src={src_v:>18,.2f}  diff={diff:>12,.2f}"
          f"   {'OK' if abs(diff) < 0.01 else 'MISMATCH'}")

print("\n" + "=" * 66)
print("6. BRIDGE LINKAGE  (activity_voucher -> voucher)")
print("=" * 66)
print(q("""
    SELECT fiscal_year,
           count(*)                     AS bridge_rows,
           count(voucher_pk)            AS matched,
           count(*) - count(voucher_pk) AS unmatched,
           round(100.0 * count(voucher_pk) / count(*), 1) AS pct_matched
    FROM activity_voucher
    GROUP BY fiscal_year
    ORDER BY fiscal_year
""").to_string(index=False))

print("\n  voucher table coverage by year:")
print(q("""
    SELECT fiscal_year, count(DISTINCT gp_lgd_code) AS gps, count(*) AS vouchers
    FROM voucher GROUP BY fiscal_year ORDER BY fiscal_year
""").to_string(index=False))

1. ROW COUNTS  (db vs expected from source frames)
  gram_panchayat                    20   OK
  plan                             204   OK
  planned_activity               12704   OK
  activity_delegation            12704   OK
  activity_asset                 12704   OK
  activity_fund                  12704   OK
  activity_training              12704   OK
  activity_community_service     12704   OK
  activity_expenditure           12730   OK
  voucher                        12440   OK
  activity_nsap                      0   (derived)
  activity_voucher                5976   (derived)

2. KEY CONSTRAINTS
                table_name constraint_type
            activity_asset     FOREIGN KEY
            activity_asset     PRIMARY KEY
activity_community_service     FOREIGN KEY
activity_community_service     PRIMARY KEY
       activity_delegation     FOREIGN KEY
       activity_delegation     PRIMARY KEY
      activity_expenditure     FOREIGN KEY
      activity_expenditure     FOREIGN KEY


In [46]:
engine.dispose()   # releases the lock, flushes to disk

In [47]:
import pandas as pd
from pathlib import Path

LOOKUP_PATH = "/mnt/c/Users/Admin/Desktop/Questions_Analysis/code_descriptions_updated.xlsx"

# what sheets are in the workbook?
xl = pd.ExcelFile(LOOKUP_PATH)
print("sheets:", xl.sheet_names)
print()

# preview each sheet: shape, columns, first rows
for s in xl.sheet_names:
    df = pd.read_excel(LOOKUP_PATH, sheet_name=s)
    print("=" * 70)
    print(f"{s}   shape={df.shape}")
    print("cols:", df.columns.tolist())
    print(df.head(5).to_string(index=False))
    print()

sheets: ['Code Descriptions', 'Summary', 'Source Files', 'Welfare Scheme Master', 'FocusArea to LSDG Theme', 'Method']

Code Descriptions   shape=(717, 8)
cols: ['variable', 'variabe_codes', 'codes_desc', 'source', 'confidence', 'your_original', 'derived_label', 'evidence']
     variable variabe_codes                      codes_desc    source confidence      your_original                   derived_label evidence
activity_type            61                    Public Works  Conflict       high    Community Works                    Public Works     1611
activity_type            62 Beneficiary Oriented Programmes   Derived     medium                NaN Beneficiary Oriented Programmes        2
   focus_area             1                     Agriculture Confirmed     medium        Agriculture                     Agriculture        1
   focus_area             2                Land improvement Confirmed       high   Land improvement                Land improvement       33
   focus_area       

In [6]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))
from config import directories

import pandas as pd
from sqlalchemy import create_engine, text

DB_PATH = directories.INTERIM_DATA / "panchayat_1.duckdb"
engine = create_engine(f"duckdb:///{DB_PATH}")
def q(sql): return pd.read_sql(sql, engine)
print(q("SHOW TABLES").to_string(index=False))

                      name
            activity_asset
activity_community_service
       activity_delegation
      activity_expenditure
             activity_fund
             activity_nsap
         activity_training
          activity_voucher
                  dim_code
            dim_lsdg_theme
        dim_welfare_scheme
            gram_panchayat
                      plan
          planned_activity
                   voucher


In [2]:
# ============================================================
# BLOCK 6 — load code description lookups
# ============================================================
import pandas as pd
from sqlalchemy import text

LOOKUP_PATH = "/mnt/c/Users/Admin/Desktop/Questions_Analysis/code_descriptions_updated.xlsx"

# ---------- read the three useful sheets ----------
cd = pd.read_excel(LOOKUP_PATH, sheet_name="Code Descriptions")
ws = pd.read_excel(LOOKUP_PATH, sheet_name="Welfare Scheme Master")
th = pd.read_excel(LOOKUP_PATH, sheet_name="FocusArea to LSDG Theme")

# ---------- dim_code: one row per (variable, code) ----------
dim_code = cd.rename(columns={
    "variabe_codes": "code",          # note: typo is in the source file
    "codes_desc": "description",
})[["variable", "code", "description", "source", "confidence"]].copy()

# codes as clean strings -> type-safe joins (same rule used throughout)
dim_code["code"] = (dim_code["code"].astype("string").str.strip()
                    .str.replace(r"\.0$", "", regex=True))
dim_code["variable"] = dim_code["variable"].astype("string").str.strip()
dim_code = (dim_code.dropna(subset=["variable", "code"])
                    .drop_duplicates(["variable", "code"]))

# ---------- welfare scheme master ----------
dim_welfare_scheme = ws.dropna(subset=["scheme_code"]).copy()
dim_welfare_scheme["scheme_code"] = (dim_welfare_scheme["scheme_code"]
    .astype("string").str.replace(r"\.0$", "", regex=True).str.strip())

# ---------- focus area -> LSDG theme ----------
dim_lsdg_theme = th.rename(columns={
    "focus area": "focus_area_name",
    "dominant LSDG theme": "lsdg_theme",
    "distinct themes seen": "distinct_themes",
    "rows": "n_rows",
}).dropna(subset=["focus_area_name"])

# ---------- create tables ----------
with engine.begin() as conn:
    for t in ["dim_code", "dim_welfare_scheme", "dim_lsdg_theme"]:
        conn.execute(text(f"DROP TABLE IF EXISTS {t}"))
    conn.execute(text("""
        CREATE TABLE dim_code (
            variable    VARCHAR,
            code        VARCHAR,
            description VARCHAR,
            source      VARCHAR,
            confidence  VARCHAR,
            PRIMARY KEY (variable, code)
        )"""))
    conn.execute(text("""
        CREATE TABLE dim_welfare_scheme (
            scheme_code VARCHAR PRIMARY KEY,
            scheme_name VARCHAR
        )"""))
    conn.execute(text("""
        CREATE TABLE dim_lsdg_theme (
            focus_area_name VARCHAR,
            lsdg_theme      VARCHAR,
            distinct_themes DOUBLE,
            n_rows          DOUBLE
        )"""))

# ---------- load ----------
dim_code.to_sql("dim_code", engine, if_exists="append", index=False)
dim_welfare_scheme.to_sql("dim_welfare_scheme", engine, if_exists="append", index=False)
dim_lsdg_theme.to_sql("dim_lsdg_theme", engine, if_exists="append", index=False)

# ---------- verify ----------
print("dim_code           :", q("SELECT count(*) FROM dim_code").iloc[0,0])
print("dim_welfare_scheme :", q("SELECT count(*) FROM dim_welfare_scheme").iloc[0,0])
print("dim_lsdg_theme     :", q("SELECT count(*) FROM dim_lsdg_theme").iloc[0,0])
print("\nvariables covered:")
print(q("SELECT variable, count(*) AS codes FROM dim_code GROUP BY 1 ORDER BY 2 DESC").to_string(index=False))

dim_code           : 717
dim_welfare_scheme : 12
dim_lsdg_theme     : 17

variables covered:
              variable  codes
     asset_subcategory    198
  asset_parameter_type    193
main_asset_subcategory    123
        asset_category     36
            focus_area     30
   fund_component_code     25
 main_asset_unit_count     25
   main_asset_category     21
      fund_scheme_code     18
           output_type      8
        operation_type      7
       activity_status      6
             work_type      4
          activity_for      4
  main_asset_unit_type      3
community_service_code      3
       asset_unit_type      3
         activity_type      2
            asset_type      2
             plan_type      2
  is_costless_activity      2
   asset_coverage_code      1
      plan_code_status      1


In [3]:
# do the decoded names line up with planned_activity's codes?
q("""
SELECT d.description AS focus_area_name, count(*) AS activities
FROM planned_activity a
JOIN dim_code d
  ON d.variable = 'focus_area'
 AND d.code = CAST(a.focus_area AS VARCHAR)
WHERE a.gp_lgd_code = '119598'
GROUP BY 1 ORDER BY 2 DESC
""")

,focus_area_name,activities
0,Sanitation,113
1,Maintenance of community system,81
2,Health,66
3,Education,38
4,Roads,37
5,Drinking water,35
6,Poverty allevation programme,22
7,Women and child development,19
8,Social welfare,14
9,Technical training and vocational education,10


In [4]:
# CHECK 1: what got loaded? which variables and how many codes each?
q("""
SELECT variable, count(*) AS n_codes
FROM dim_code
GROUP BY 1 ORDER BY 2 DESC
""")

,variable,n_codes
0,asset_subcategory,198
1,asset_parameter_type,193
2,main_asset_subcategory,123
3,asset_category,36
4,focus_area,30
5,fund_component_code,25
6,main_asset_unit_count,25
7,main_asset_category,21
8,fund_scheme_code,18
9,output_type,8


In [5]:
# CHECK 2: do the codes actually decode? Andhrua's activities by focus area name
q("""
SELECT d.description AS focus_area_name,
       count(*) AS activities,
       sum(a.total_cost) AS planned_cost
FROM planned_activity a
LEFT JOIN dim_code d
  ON d.variable = 'focus_area'
 AND d.code = CAST(a.focus_area AS VARCHAR)
WHERE a.gp_lgd_code = '119598'
GROUP BY 1
ORDER BY activities DESC
""")

,focus_area_name,activities,planned_cost
0,Sanitation,113,6022760.0
1,Maintenance of community system,81,1186584.0
2,Health,66,283500.0
3,Education,38,2667573.0
4,Roads,37,5426747.0
5,Drinking water,35,4489468.0
6,Poverty allevation programme,22,NaN
7,Women and child development,19,495000.0
8,Social welfare,14,837915.0
9,Land improvement,10,625413.0


In [7]:
engine.dispose()
print("engine disposed — database lock released")

engine disposed — database lock released


In [8]:
import pandas as pd

# 1. what tables actually exist in the file right now?
print(q("SHOW TABLES").to_string(index=False))

                      name
            activity_asset
activity_community_service
       activity_delegation
      activity_expenditure
             activity_fund
             activity_nsap
         activity_training
          activity_voucher
                  dim_code
            dim_lsdg_theme
        dim_welfare_scheme
            gram_panchayat
                      plan
          planned_activity
                   voucher


In [9]:
# 2. explicitly test for each dim table
for t in ["dim_code", "dim_welfare_scheme", "dim_lsdg_theme"]:
    try:
        n = q(f"SELECT count(*) FROM {t}").iloc[0, 0]
        print(f"{t:22} EXISTS  ({n} rows)")
    except Exception as e:
        print(f"{t:22} MISSING — {str(e)[:60]}")

dim_code               EXISTS  (717 rows)
dim_welfare_scheme     EXISTS  (12 rows)
dim_lsdg_theme         EXISTS  (17 rows)


In [10]:
# does it actually decode? Andhrua by focus area name
q("""
SELECT d.description AS focus_area_name, count(*) AS activities
FROM planned_activity a
LEFT JOIN dim_code d
  ON d.variable = 'focus_area'
 AND d.code = CAST(a.focus_area AS VARCHAR)
WHERE a.gp_lgd_code = '119598'
GROUP BY 1 ORDER BY 2 DESC
""")

,focus_area_name,activities
0,Sanitation,113
1,Maintenance of community system,81
2,Health,66
3,Education,38
4,Roads,37
5,Drinking water,35
6,Poverty allevation programme,22
7,Women and child development,19
8,Social welfare,14
9,Land improvement,10


In [11]:
import pandas as pd
from pathlib import Path

print("engine URL :", engine.url)          # the file this engine writes to

# resolve it to an absolute path
db = str(engine.url).replace("duckdb:///", "")
p = Path(db)
print("path       :", p.resolve())
print("exists     :", p.exists())
print("size       :", f"{p.stat().st_size/1e6:.2f} MB")
print("modified   :", pd.Timestamp(p.stat().st_mtime, unit='s'))

# prove dim_code is in THIS file
print("\ntables here:", q("SHOW TABLES")["name"].tolist())

engine URL : duckdb:////home/admin_abhi/GitHub/ai-for-panchayats/data/interim/panchayat_1.duckdb
path       : /home/admin_abhi/GitHub/ai-for-panchayats/data/interim/panchayat_1.duckdb
exists     : True
size       : 19.41 MB
modified   : 2026-08-09 08:57:20.987946510

tables here: ['activity_asset', 'activity_community_service', 'activity_delegation', 'activity_expenditure', 'activity_fund', 'activity_nsap', 'activity_training', 'activity_voucher', 'dim_code', 'dim_lsdg_theme', 'dim_welfare_scheme', 'gram_panchayat', 'plan', 'planned_activity', 'voucher']


In [12]:
from sqlalchemy import text

with engine.begin() as conn:
    conn.execute(text("CHECKPOINT"))     # flush WAL into the .duckdb file
print("checkpointed")

engine.dispose()                          # then close cleanly

checkpointed


In [13]:
import duckdb
con = duckdb.connect(
    "/home/admin_abhi/GitHub/ai-for-panchayats/data/interim/panchayat_1.duckdb",
    read_only=True)
print(con.sql("SHOW TABLES"))
con.close()

ConnectionException: Connection Error: Can't open a connection to same database file with a different configuration than existing connections